# LSTM du doan future return 5 ngay
Notebook nay duoc thiet ke de chay tren Google Colab hoac local VS Code.

LSTM khong bat buoc GPU voi mo hinh nho va du lieu daily, nhung GPU Colab thuong giup train nhanh hon. Hay vao Runtime > Change runtime type > T4 GPU neu tai khoan co san. Notebook dung target `future_return_5d`, sequence 30 phien va split theo thoi gian.

In [ ]:
%pip -q install tensorflow scikit-learn pandas matplotlib joblib

import os
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)

## Cau hinh du lieu

Tren Colab, dat file `features_all.csv` trong Google Drive va sua `DATA_PATH` neu can. Neu upload thu cong, co the dung `files.upload()` thay cho Google Drive.

In [ ]:
RUNNING_IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
USE_GOOGLE_DRIVE = False

if RUNNING_IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = Path('/content/drive/MyDrive/HQTCSDL_stocks/data/clean/features_all.csv')
else:
    DATA_PATH = Path('data/clean/features_all.csv')

OUTPUT_DIR = Path('lstm_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEQ_LEN = 30
HORIZON = 5
TRAIN_END = '2024-01-01'
VALIDATION_END = '2025-01-01'
MAX_SEQUENCES = {'train': 180_000, 'validation': 60_000, 'test': 80_000}

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Khong tim thay {DATA_PATH}. Hay sua DATA_PATH o cell nay.')
print('DATA_PATH:', DATA_PATH.resolve())
print('Output:', OUTPUT_DIR.resolve())

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else 'Khong co GPU, se chay bang CPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = [str(c).strip().lower() for c in df.columns]
df['trading_date'] = pd.to_datetime(df['trading_date'], errors='coerce')
df['symbol'] = df['symbol'].astype(str).str.strip().str.upper()
df = df.sort_values(['symbol', 'trading_date']).drop_duplicates(['symbol', 'trading_date'])
df = df.dropna(subset=['symbol', 'trading_date', 'close'])

candidate_features = [
    'open', 'high', 'low', 'close', 'volume', 'encode_sector',
    'return_1d', 'return_3d', 'return_5d', 'return_10d', 'return_20d',
    'price_vs_ma20', 'ma5_vs_ma20', 'volatility_5d', 'volatility_20d',
    'volatility_change', 'drawdown_20d', 'volume_ratio_5_20',
    'volume_change_1d', 'daily_range', 'body_ratio', 'close_position'
]
FEATURES = [c for c in candidate_features if c in df.columns]
for column in FEATURES:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['future_return_5d'] = df.groupby('symbol')['close'].shift(-HORIZON) / df['close'] - 1.0
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + ['future_return_5d'])
print('Rows:', len(df), '| Symbols:', df['symbol'].nunique())
print('Features:', FEATURES)
df.head()

## Split theo thoi gian va scale
Scaler chi duoc fit tren cac dong train. Sequence duoc tao rieng cho tung symbol, sau do gan theo ngay cua target; vi vay khong co sequence noi hai ma khac nhau.

In [ ]:
train_end = pd.Timestamp(TRAIN_END)
validation_end = pd.Timestamp(VALIDATION_END)
train_rows = df['trading_date'] < train_end
validation_rows = (df['trading_date'] >= train_end) & (df['trading_date'] < validation_end)
test_rows = df['trading_date'] >= validation_end

scaler = RobustScaler()
scaler.fit(df.loc[train_rows, FEATURES])
df_scaled = df.copy()
df_scaled[FEATURES] = scaler.transform(df_scaled[FEATURES]).astype('float32')
joblib.dump(scaler, OUTPUT_DIR / 'scaler.joblib')

print('Train rows:', train_rows.sum())
print('Validation rows:', validation_rows.sum())
print('Test rows:', test_rows.sum())

In [ ]:
def make_sequences(frame, feature_columns, sequence_length, split_masks, max_sequences):
    sequences = {name: [] for name in split_masks}
    targets = {name: [] for name in split_masks}
    target_dates = {name: [] for name in split_masks}
    symbols = {name: [] for name in split_masks}
    for symbol, group in frame.groupby('symbol', sort=False):
        group = group.sort_values('trading_date')
        values = group[feature_columns].to_numpy(dtype='float32')
        y = group['future_return_5d'].to_numpy(dtype='float32')
        dates = group['trading_date'].to_numpy()
        for end in range(sequence_length - 1, len(group)):
            target_date = pd.Timestamp(dates[end])
            split_name = next((name for name, mask in split_masks.items() if mask(target_date)), None)
            if split_name is None or len(sequences[split_name]) >= max_sequences[split_name]:
                continue
            sequences[split_name].append(values[end - sequence_length + 1:end + 1])
            targets[split_name].append(y[end])
            target_dates[split_name].append(target_date)
            symbols[split_name].append(symbol)
    result = {}
    for name in split_masks:
        result[name] = (
            np.asarray(sequences[name], dtype='float32'),
            np.asarray(targets[name], dtype='float32'),
            pd.to_datetime(target_dates[name]),
            np.asarray(symbols[name])
        )
        print(name, result[name][0].shape)
    return result

split_masks = {
    'train': lambda date: date < train_end,
    'validation': lambda date: train_end <= date < validation_end,
    'test': lambda date: date >= validation_end,
}
data = make_sequences(df_scaled, FEATURES, SEQ_LEN, split_masks, MAX_SEQUENCES)
X_train, y_train, _, _ = data['train']
X_val, y_val, _, _ = data['validation']
X_test, y_test, test_dates, test_symbols = data['test']
if min(len(X_train), len(X_val), len(X_test)) == 0:
    raise ValueError('Mot trong cac tap khong co sequence. Hay kiem tra moc thoi gian.')

In [ ]:
tf.keras.backend.clear_session()
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN, len(FEATURES))),
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1)
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.Huber(),
    metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')]
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(OUTPUT_DIR / 'best_lstm.keras', monitor='val_loss', save_best_only=True)
]
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)
model.save(OUTPUT_DIR / 'model_lstm.keras')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.xlabel('Epoch')
plt.ylabel('Huber loss')
plt.title('LSTM training history')
plt.legend()
plt.show()

pred = model.predict(X_test, batch_size=1024, verbose=0).reshape(-1)
baseline = np.zeros_like(y_test)
metrics = {
    'mae': float(mean_absolute_error(y_test, pred)),
    'rmse': float(np.sqrt(mean_squared_error(y_test, pred))),
    'r2': float(r2_score(y_test, pred)),
    'directional_accuracy_pct': float((np.sign(y_test) == np.sign(pred)).mean() * 100),
    'baseline_zero_return_mae': float(mean_absolute_error(y_test, baseline)),
    'baseline_zero_return_rmse': float(np.sqrt(mean_squared_error(y_test, baseline))),
}
print(json.dumps(metrics, indent=2))
with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as file:
    json.dump(metrics, file, indent=2)

predictions = pd.DataFrame({
    'trading_date': test_dates,
    'symbol': test_symbols,
    'actual_future_return_5d': y_test,
    'predicted_future_return_5d': pred,
})
predictions['actual_direction'] = (predictions['actual_future_return_5d'] > 0).astype(int)
predictions['predicted_direction'] = (predictions['predicted_future_return_5d'] > 0).astype(int)
predictions.to_csv(OUTPUT_DIR / 'predictions.csv', index=False)
predictions.head()

In [ ]:
plot_df = predictions.sort_values('trading_date').head(500)
plt.figure(figsize=(14, 4))
plt.plot(plot_df['actual_future_return_5d'].to_numpy(), label='actual', alpha=0.8)
plt.plot(plot_df['predicted_future_return_5d'].to_numpy(), label='predicted', alpha=0.8)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Actual vs predicted future return - first 500 test samples')
plt.legend()
plt.show()

print('Da luu:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path)

## Cach so sanh voi Model 2

So sanh LSTM voi Model 2 chi hop le khi dung cung `future_return_5d`, cung khoang test va cung chi phi giao dich. Hay doi chieu `metrics.json` cua notebook voi `models/model2/reports/metrics.json`, sau do nen bo sung rank IC, Precision@K, cumulative return, Sharpe va maximum drawdown. Neu LSTM chi tot hon MAE nhung khong tot hon Directional Accuracy hoac backtest, khong nen chon LSTM cho trading.

De chay tren Colab: upload notebook, bat GPU, upload `features_all.csv` vao Drive, sua `USE_GOOGLE_DRIVE = True` va sua `DATA_PATH` cho dung duong dan.